# Day 37 – Q1: TF-IDF from Scratch

Full TF-IDF implementation without sklearn on ShopSense Reviews (10K rows).

In [ ]:
import numpy as np
import pandas as pd
import re
from collections import Counter
from scipy.sparse import lil_matrix, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

REVIEWS_PATH = "shopsense_reviews-2.csv"
QUERY = "wireless earbuds battery life poor"
CATEGORY = "Electronics"


In [ ]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns and "category" in df.columns
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")
    except AssertionError:
        raise ValueError("Expected columns 'review_text' and 'category' not found.")

df = load_reviews(REVIEWS_PATH)
print("Shape:", df.shape)
print("Categories:", df["category"].unique())
print("Null review_text:", df["review_text"].isna().sum())


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(text.split())

def tokenize(text):
    return clean_text(text).split()

df["tokens"] = df["review_text"].apply(tokenize)
df["cleaned"] = df["tokens"].apply(" ".join)
print("Sample tokens:", df["tokens"].iloc[3])


In [ ]:
def build_vocabulary(token_series):
    vocab = sorted(set(tok for tokens in token_series for tok in tokens))
    return {word: idx for idx, word in enumerate(vocab)}

vocab = build_vocabulary(df["tokens"])
print("Vocab size:", len(vocab))


In [ ]:
def compute_tf(tokens):
    count = Counter(tokens)
    total = len(tokens) if tokens else 1
    return {word: cnt / total for word, cnt in count.items()}

def compute_idf(token_series, vocab):
    N = len(token_series)
    df_counts = Counter()
    for tokens in token_series:
        for word in set(tokens):
            df_counts[word] += 1
    idf = {word: np.log((N + 1) / (df_counts.get(word, 0) + 1)) + 1
           for word in vocab}
    return idf


In [ ]:
idf = compute_idf(df["tokens"], vocab)
print("IDF computed for vocab of size:", len(idf))


In [ ]:
def build_tfidf_matrix(token_series, vocab, idf):
    V = len(vocab)
    N = len(token_series)
    matrix = lil_matrix((N, V), dtype=np.float32)
    for i, tokens in enumerate(token_series):
        tf = compute_tf(tokens)
        for word, tf_val in tf.items():
            if word in vocab:
                matrix[i, vocab[word]] = tf_val * idf[word]
    return csr_matrix(matrix)

tfidf_matrix = build_tfidf_matrix(df["tokens"], vocab, idf)
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Non-zero elements:", tfidf_matrix.nnz)


## (b) Top-5 Reviews for Query

In [ ]:
def cosine_similarity_sparse(matrix, query_vec):
    dot = matrix.dot(query_vec)
    norm_docs = np.array(matrix.power(2).sum(axis=1)).flatten() ** 0.5
    norm_q = np.sqrt(query_vec.dot(query_vec))
    denom = norm_docs * norm_q
    denom[denom == 0] = 1e-10
    return dot / denom

def rank_top_k(query_text, tfidf_matrix, vocab, idf, df, k=5):
    q_tokens = tokenize(query_text)
    tf_q = compute_tf(q_tokens)
    q_vec = np.zeros(len(vocab), dtype=np.float32)
    for word, tf_val in tf_q.items():
        if word in vocab:
            q_vec[vocab[word]] = tf_val * idf[word]
    scores = cosine_similarity_sparse(tfidf_matrix, q_vec)
    top_k_idx = np.argsort(scores)[-k:][::-1]
    results = df.iloc[top_k_idx][["review_id", "category", "review_text", "rating"]].copy()
    results["cosine_score"] = scores[top_k_idx].round(4)
    return results

top5 = rank_top_k(QUERY, tfidf_matrix, vocab, idf, df)
print("Query:", QUERY)
print(top5[["review_id", "category", "cosine_score", "review_text"]].to_string())


## (c) Compare with sklearn TfidfVectorizer

In [ ]:
def sklearn_tfidf(texts):
    vec = TfidfVectorizer()
    matrix = vec.fit_transform(texts)
    return matrix, vec

sk_matrix, sk_vec = sklearn_tfidf(df["cleaned"].tolist())
print("sklearn TF-IDF shape:", sk_matrix.shape)


In [ ]:
def average_l2_difference(scratch_matrix, sk_matrix, vocab, sk_vec, sample_n=500):
    sk_vocab = sk_vec.vocabulary_
    common_words = [w for w in vocab if w in sk_vocab][:sample_n]
    scratch_cols = [vocab[w] for w in common_words]
    sk_cols = [sk_vocab[w] for w in common_words]
    scratch_sub = scratch_matrix[:, scratch_cols].toarray()
    sk_sub = sk_matrix[:, sk_cols].toarray()
    l2_diffs = np.linalg.norm(scratch_sub - sk_sub, axis=1)
    return l2_diffs.mean()

avg_l2 = average_l2_difference(tfidf_matrix, sk_matrix, vocab, sk_vec)
print(f"Average L2 difference (scratch vs sklearn): {avg_l2:.4f}")
print("Note: Difference exists due to sklearn using L2-normalized output by default.")


## (d) Highest Average TF-IDF Word in Electronics

In [ ]:
def top_word_by_category(df, tfidf_matrix, vocab, category):
    mask = df["category"] == category
    sub_matrix = tfidf_matrix[mask.values]
    mean_scores = np.array(sub_matrix.mean(axis=0)).flatten()
    idx_to_word = {v: k for k, v in vocab.items()}
    top_idx = int(np.argmax(mean_scores))
    return idx_to_word[top_idx], mean_scores[top_idx]

top_word, top_score = top_word_by_category(df, tfidf_matrix, vocab, CATEGORY)
print(f"Highest avg TF-IDF word in '{CATEGORY}': '{top_word}' | Score: {top_score:.4f}")
print("Explanation: This word appears frequently within Electronics reviews but")
print("rarely in other categories, giving it high TF in those docs and a moderate IDF overall.")
